# STEP 1: Install dependencies

In [3]:
!pip install -q crewai google-generativeai duckduckgo_search yfinance pandas beautifulsoup4 python-dotenv
!pip install -q crewai-tools

# STEP 2: Imports

In [4]:
import os
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, LLM
import yfinance as yf
from duckduckgo_search import DDGS
from crewai.tools import BaseTool

# Load GEMINI API key (replace with your key or set in .env)

In [5]:
from getpass import getpass
os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API Key: ")

Enter your Gemini API Key: ··········


# STEP 3: Setup Gemini LLM via CrewAI

In [6]:
llm = LLM(
    model="gemini/gemini-2.0-flash",  # Ensure this model is valid and accessible
    api_key=os.environ["GEMINI_API_KEY"],
    temperature=0.7
)

# STEP 4: Define Tools

In [7]:
class StockSearchTool(BaseTool):
    name: str = "StockNewsSearcher"
    description: str = "Search for the latest news and updates about a stock using DuckDuckGo"

    def _run(self, query: str) -> str:
        with DDGS() as ddgs:
            results = ddgs.text(query, max_results=2)  # Limit to 2 results to reduce output
            return "\n".join([r['body'] for r in results])

class YahooFinanceTool(BaseTool):
    name: str = "YahooFinanceFetcher"
    description: str = "Get the latest 1-month stock price history for a given ticker using yFinance"

    def _run(self, ticker: str) -> str:
        stock = yf.Ticker(ticker)
        hist = stock.history(period="1mo")
        return hist.tail(3).to_string()  # Limit to last 3 rows for concise output

# STEP 5: Define Agents

In [8]:
stock_analyst = Agent(
    role='Stock Analyst',
    goal='Analyze recent stock data and news',
    backstory='Expert in financial trends, macro indicators, and company performance',
    verbose=True,  # Keep agent verbose for debugging, but we'll adjust Crew verbose
    allow_delegation=False,
    llm=llm
)

report_writer = Agent(
    role='Report Generator',
    goal='Write investor-friendly summaries of stock analysis',
    backstory='Professional writer with expertise in finance reporting',
    verbose=True,
    allow_delegation=False,
    llm=llm
)

# STEP 6: Create Tasks

In [9]:
search_tool = StockSearchTool()
finance_tool = YahooFinanceTool()

search_task = Task(
    description="Search latest news and updates about the stock 'AAPL' using DuckDuckGo.",
    expected_output="Summarized news highlights for Apple stock.",
    agent=stock_analyst,
    tools=[search_tool]
)

analysis_task = Task(
    description="Analyze Apple stock price trends using yFinance.",
    expected_output="Key trends and technical highlights for the past month.",
    agent=stock_analyst,
    tools=[finance_tool]
)

report_task = Task(
    description="Write a clean investor report using previous analysis and news insights.",
    expected_output="Concise report with market summary and investment outlook.",
    agent=report_writer
)

# STEP 7: Assemble the Crew

In [10]:
crew = Crew(
    agents=[stock_analyst, report_writer],
    tasks=[search_task, analysis_task, report_task],
    verbose=False  # Set to False to reduce rich console output and avoid RecursionError
)

# STEP 8: Run the Agent Crew

In [11]:
result = crew.kickoff()

# STEP 10: Output the result
print("\n📊 Final Stock Analysis Report:\n")
print(result)

# Agent: Stock Analyst
## Task: Search latest news and updates about the stock 'AAPL' using DuckDuckGo.


# Agent: Stock Analyst
## Thought: I need to find the latest news and updates about Apple stock (AAPL) using the StockNewsSearcher tool.
## Using tool: StockNewsSearcher
## Tool Input: 
"{\"query\": \"AAPL stock news\"}"
## Tool Output: 
Get the latest Apple Inc. (AAPL) stock news and headlines to help you in your trading and investing decisions.
AAPL: Apple Inc - Stock Price, Quote and News - CNBC


# Agent: Stock Analyst
## Final Answer: 
Get the latest Apple Inc. (AAPL) stock news and headlines to help you in your trading and investing decisions.
AAPL: Apple Inc - Stock Price, Quote and News - CNBC


# Agent: Stock Analyst
## Task: Analyze Apple stock price trends using yFinance.


# Agent: Stock Analyst
## Thought: I need to analyze the Apple stock price trends for the past month. I can use the YahooFinanceFetcher tool to get the stock price history.
## Using tool: YahooFinance